# Information retrieval: from word matching to learned embeddings

**Goal:** given a scientific claim, rank abstracts that provide relevant evidence.
We will compare **BM25, TF-IDF, a custom Transformer, MiniLM, MiniLM + LoRA,
and EmbeddingGemma** on the same queries and documents.

The lesson asks three questions:
1. How far can matching words take us?
2. What changes when we learn representations from relevance labels?
3. What do pretraining and a small task-specific update add?

Use this repository's Python environment. From the repository root:
```bash
uv sync --locked
uv run jupyter lab text_ir/teaching.ipynb
```
Run cells from top to bottom. **Run All trains both models by default**: 50 epochs
for the custom model and 20 for LoRA. Both keep the weights from the best validation
epoch in `artifacts/teaching/`. Set `TRAIN_MODELS = False` to reuse weights from a
previous notebook run. Repeating training replaces these files.

The [custom model and training loop](model.py) and
[MiniLM adapter](lora.py) keep the notebook focused on retrieval ideas.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "text_ir/model.py").is_file())
sys.path.insert(0, str(ROOT))

from text_ir.data import read_dataset
from text_ir.evaluate import evaluate_holdout, ranking_metrics
from text_ir.model import select_device, train_model

DATA = ROOT / "data/scifact"
SPLITS = ROOT / "text_ir/splits.json"
OUTPUT = ROOT / "artifacts/teaching"
CACHE = ROOT / ".cache/models"
OUTPUT.mkdir(parents=True, exist_ok=True)
TRAIN_MODELS = True  # False reuses weights saved by a previous run of this notebook.
device = select_device()

## 0. Download the data

**Task:** run the cell below to download SciFact's abstracts, claims, and training
relevance labels into `data/scifact/`. The first download needs internet access;
subsequent runs reuse the local files.

The cell also prepares the recorded training/validation split used throughout this
lesson. The two file lists in `text_ir/splits.json` are reused, so rerunning this
step keeps the same queries in each group. The official test judgments are not used.

In [ ]:
from text_ir.data import prepare_scifact, prepare_splits

prepare_scifact(DATA)  # Download only if required files are missing.
prepare_splits(DATA, SPLITS)
print(f"Data ready: {DATA}\nTrain/validation split: {SPLITS}")

## 1. One retrieval task for every method

SciFact pairs scientific claims with relevant abstracts. Evidence can **support or
refute** a claim: retrieval finds evidence, rather than deciding whether a claim is true.

Use the recorded, roughly 80/20 training/validation split. All methods search the same complete
corpus, using document titles plus abstracts. Training sees only training labels.
The validation set selects the best training epoch, so these are **validation
results**, not an untouched test benchmark. The official test set stays unused.

In [ ]:
documents, splits = read_dataset(DATA, SPLITS)
validation = splits["holdout"]
document_ids, query_ids = sorted(documents), sorted(validation)
document_texts = [documents[did] for did in document_ids]
query_texts = [validation[qid]["text"] for qid in query_ids]

print(f"{len(documents):,} documents; {len(splits['train']):,} training queries; {len(validation):,} validation queries")
print("Example claim:", query_texts[0])

## 2. What counts as a good ranking?

We inspect the **top 10 results** and average each metric equally across queries.
All three range from 0 to 1; higher is better.

| Metric | Intuition |
|---|---|
| **Recall@10** | What fraction of the known relevant documents did we find? |
| **MRR@10** | How soon did we find the first relevant document? Use `1 / its rank`, or 0 if absent. |
| **nDCG@10** | How much relevant evidence appears near the top? Discount later hits and divide by the ideal ranking. |

With exactly one relevant document at rank 3, Recall@10 is **1**, MRR@10 is **1/3**,
and nDCG@10 is **1/log₂(4) = 0.5**. Finding a document and ranking it first are different goals.

Here **MRR** means *mean reciprocal rank*. **MMR** means *maximal marginal relevance*,
a separate method for diversifying results, rather than the evaluation metric used here.
Unjudged documents count as zero for scoring; some may still contain useful evidence.

In [ ]:
METRICS = ["nDCG@10", "Recall@10", "MRR@10"]
results = {}


def evaluate_scores(scores):
    """Rows are queries; columns are documents in document_ids order."""
    top_indices = np.argsort(-scores, axis=1, kind="stable")[:, :10]
    rows = [
        ranking_metrics([document_ids[i] for i in indices], set(validation[qid]["positive_document_ids"]))
        for qid, indices in zip(query_ids, top_indices)
    ]
    return pd.DataFrame(rows).mean().to_dict()

## 3. What is BM25?

BM25 scores a document by adding up evidence from its matching query words:

- **Rare words matter more.** A specific protein name is usually more informative than “study.”
- **Repeated matches have diminishing returns.** Ten repetitions do not give ten times the credit.
- **Document length matters.** Long documents have more opportunities to match by chance.

Think of it as a carefully weighted word-matching score. No neural network or relevance-label
training is needed. It can work well for exact scientific terminology, but a synonym with no
matching word receives no direct credit.

We use positive-IDF BM25 with `k1=1.2` (frequency saturation) and `b=0.75` (length
normalization). Each distinct query word contributes once. Both lexical methods use
lowercase word tokens, with no stemming or stop-word removal.
[BM25 background](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from text_ir.evaluate import bm25_weights

# Learn the vocabulary and word statistics from documents, not relevance labels.
vectorizer = CountVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b")
document_counts = vectorizer.fit_transform(document_texts)
query_counts = vectorizer.transform(query_texts)

bm25_documents = bm25_weights(document_counts, k1=1.2, b=0.75)
results["BM25"] = evaluate_scores((query_counts.sign() @ bm25_documents.T).toarray())

## 4. What is TF-IDF?

Represent each document as a vector with one coordinate per vocabulary word.
A word's weight combines **term frequency** (how often it appears in this document)
with **inverse document frequency** (how rare it is across documents).

Normalize each vector to length 1. Its dot product with a normalized query vector is
**cosine similarity**: a score for how closely their weighted word patterns match.
Our implementation uses raw counts and smoothed IDF, `log((1 + N)/(1 + df)) + 1`.

TF-IDF provides a simple vector-space baseline. BM25 adds an explicit saturating
frequency term and document-length adjustment. Both still depend on word overlap.
[TF-IDF implementation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html).

In [ ]:
tfidf = TfidfTransformer(norm="l2")
document_tfidf = tfidf.fit_transform(document_counts)
query_tfidf = tfidf.transform(query_counts)
results["TF-IDF"] = evaluate_scores((query_tfidf @ document_tfidf.T).toarray())

## 5. Train a custom bidirectional attention model

Replace the word-count vector with a **learned embedding**:

**tokens + positions → bidirectional Transformer → weighted pooling → normalized vector**

Each token can use words on both sides to build its representation. Learned pooling
combines the token states into one vector. The same encoder processes queries and
documents; similarity is their dot product. Documents are encoded independently of
the query, so their vectors can be indexed in advance.

Training pulls a query toward known relevant documents and pushes competing candidates
away. The shared training function uses a contrastive loss and handles multiple known
positives. Unjudged candidates are useful training competitors, but are not guaranteed irrelevant.

Our small model starts from **random weights**: 2 Transformer layers, 4 heads,
128-dimensional token states and retrieval vectors, with a 256-token input cap.
A pretrained tokenizer supplies tokenization rules, not pretrained semantic knowledge.
We allow 50 epochs and keep the epoch with the best validation nDCG@10;
more epochs do not guarantee a better model.

In [ ]:
from text_ir.model import TextIREncoder
from transformers import AutoTokenizer

torch.manual_seed(42)
custom_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
custom = TextIREncoder(
    len(custom_tokenizer), custom_tokenizer.pad_token_id,
    hidden_dim=128, num_heads=4, num_layers=2, embedding_dim=128,
).to(device)
custom_path = OUTPUT / "custom.pt"
if TRAIN_MODELS:
    train_model(custom, custom_tokenizer, documents, splits, device,
                epochs=50, batch_size=16, learning_rate=3e-4, checkpoint=custom_path)
custom.load_state_dict(torch.load(custom_path, map_location="cpu", weights_only=True))
_ = custom.eval()

In [ ]:
results["Custom model"] = evaluate_holdout(custom, custom_tokenizer, documents, validation, device)
del custom, custom_tokenizer  # Evaluate one neural model at a time.

## 6. Train MiniLM with LoRA

**MiniLM** already maps text into a useful embedding space. We start from
[all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
a pretrained encoder further trained on a large collection of sentence pairs.
Its frozen form is our **MiniLM** baseline.

**LoRA** adapts selected weight matrices while keeping the original weights frozen:

$$W_{adapted} = W_{frozen} + (\alpha/r)BA$$

The two small matrices `A` and `B` form a low-rank update. Here `r=8`, and the updates
are attached to the attention query/value projections. Only these updates learn from
our training queries. The encoder uses mean pooling and normalized 384-dimensional
vectors, with the same 256-token cap as our custom model.

Use the same retrieval objective as the custom model, allow 20 epochs, and select the
best validation epoch. Comparing the adapter **off versus on** keeps the backbone,
tokenizer, pooling, and input budget fixed. LoRA may help, but improvement must be measured.
[Original LoRA paper](https://arxiv.org/abs/2106.09685).

In [ ]:
from text_ir.lora import create_minilm

torch.manual_seed(42)
minilm, minilm_tokenizer = create_minilm(CACHE, rank=8, alpha=16)
minilm = minilm.to(device)
lora_path = OUTPUT / "lora.pt"
if TRAIN_MODELS:
    train_model(minilm, minilm_tokenizer, documents, splits, device,
                epochs=20, batch_size=8, learning_rate=1e-4, checkpoint=lora_path)
# Only adapter weights are saved; the frozen backbone comes from MiniLM.
minilm.load_state_dict(torch.load(lora_path, map_location="cpu", weights_only=True), strict=False)
_ = minilm.eval()

In [ ]:
with minilm.encoder.disable_adapter():
    results["MiniLM"] = evaluate_holdout(minilm, minilm_tokenizer, documents, validation, device)
results["MiniLM + LoRA"] = evaluate_holdout(minilm, minilm_tokenizer, documents, validation, device)
del minilm, minilm_tokenizer

## 7. A stronger pretrained reference: EmbeddingGemma

[EmbeddingGemma-300M](https://ai.google.dev/gemma/docs/embeddinggemma/model_card)
is a pretrained embedding model. We use it frozen: no local training is performed.
Its retrieval prefixes tell the model whether an input is a search query or a document.

Gemma uses 768-dimensional embeddings and up to 2,048 input tokens. It differs from
our other models in training data, size, tokenizer, and input budget. This is a
**practical comparison of retrievers**, rather than an experiment isolating attention.
All methods receive the same document bodies; lexical methods use the full text,
while each neural encoder applies its own token limit.

Before the first download, accept the [model access terms](https://huggingface.co/google/embeddinggemma-300m)
and run `uv run hf auth login` in a terminal. We pin the model revision and reuse the
project's download cache. Encoding the whole corpus can take several minutes.

In [ ]:
from sentence_transformers import SentenceTransformer

gemma = SentenceTransformer(
    "google/embeddinggemma-300m",
    revision="57c266a740f537b4dc058e1b0cda161fd15afa75",
    cache_folder=str(CACHE),
    device=str(device), model_kwargs={"dtype": torch.float32},
)
gemma_documents = gemma.encode(
    document_texts, prompt="title: none | text: ", batch_size=8,
    normalize_embeddings=True, show_progress_bar=True,
)
gemma_queries = gemma.encode(
    query_texts, prompt="task: search result | query: ", batch_size=8,
    normalize_embeddings=True, show_progress_bar=True,
)
results["EmbeddingGemma"] = evaluate_scores(gemma_queries @ gemma_documents.T)
del gemma

## 8. Compare the six retrievers

Every row below searches the full corpus for the same validation queries. Higher
is better in every column. Compare ranks and relevance metrics across models;
raw similarity scores are not calibrated across different embedding spaces.

In [ ]:
comparison = pd.DataFrame.from_dict(results, orient="index")[METRICS]
comparison.index.name = "Method"
display(comparison.style.format("{:.3f}"))
comparison.to_csv(OUTPUT / "comparison.csv")

figure, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True, layout="constrained")
colors = ["#0072B2", "#E69F00", "#009E73"]
for axis, metric, color in zip(axes, METRICS, colors):
    bars = axis.barh(comparison.index, comparison[metric], color=color)
    axis.bar_label(bars, fmt="%.3f", padding=3)
    axis.set(title=metric, xlim=(0, 1.12))
    axis.set_xticks([0, 0.25, 0.5, 0.75, 1])
axes[0].invert_yaxis()
figure.suptitle("Same queries, same corpus — validation results")
figure.savefig(OUTPUT / "comparison.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. What should we learn from the comparison?

- **Word matching is a serious baseline.** Scientific names and exact terms can make lexical methods competitive.
- **Attention supplies a capability; training makes it useful.** A custom Transformer has to learn useful similarities from a small local dataset.
- **Pretraining changes the starting point.** MiniLM and Gemma bring representations learned before this experiment.
- **LoRA tests adaptation.** Its gain or loss over frozen MiniLM measures whether our small update helps this validation task.
- **Evaluate relevance, not the appearance of a vector space.** A round PCA cloud or mean cosine near zero does not guarantee good retrieval.

**Discuss:** Which method would you choose for finding one good result? For recovering
all known evidence? Does LoRA improve all three metrics? What extra information about
latency, memory, and training cost would change your choice?

These checkpoints were selected using this validation set, and results come from one
training run. Public pretraining overlap with SciFact cannot be ruled out. Freeze choices
before using the official test set for a final evaluation.